# TetheredAI MLB Full Self-Contained API → Features → Models → Predictions Lab

This notebook is intentionally **self-contained**. It does not import your project scripts such as `feature_engineering.py`, `01_fetch_odds.py`, or `05_score_today.py`.

It is meant to be the central model-development workbench while you are still iterating on features and modeling logic. Once a feature/model is proven here, port the stable logic back into production scripts/jobs.

## What this notebook does inline

1. Pulls MLB schedule/results directly from the MLB Stats API.
2. Pulls sportsbook odds directly from The Odds API.
3. Pulls Statcast pitch-level data directly via `pybaseball.statcast` / Baseball Savant.
4. Engineers team, starter, bullpen, pitch-mix, batted-ball, ELO, totals, and margin features inline.
5. Runs EDA, missingness, feature inventory, leakage audits, correlation checks, and optional PCA.
6. Trains/evaluates:
   - Moneyline classification model: `target_home_win`
   - Total-runs regression/Poisson-style model: `target_total_runs`
   - Home-margin/run-line model: `target_home_margin`
7. Selects champion candidates with leakage guardrails.
8. Scores upcoming games and creates Streamlit-ready prediction CSVs.
9. Optionally exports model bundles and uploads artifacts to GCS.

> This notebook is a lab version, not a byte-for-byte copy of your production scripts. It is designed to be readable, editable, and complete in one place.


## 0. Setup and package installs

Run this first. In Colab/Colab Enterprise, uncomment the `%pip install` line if packages are missing.


In [ ]:
# Uncomment if running in a fresh Colab runtime.
# %pip install -q pandas numpy scikit-learn scipy requests joblib pyarrow pybaseball xgboost lightgbm google-cloud-storage matplotlib

from __future__ import annotations

import json
import math
import os
import time
import warnings
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import requests

from scipy.stats import norm
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, PoissonRegressor
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

try:
    import joblib
    HAS_JOBLIB = True
except Exception:
    HAS_JOBLIB = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
print("Setup complete")
print("XGBoost:", HAS_XGBOOST, "LightGBM:", HAS_LIGHTGBM, "joblib:", HAS_JOBLIB)


## 1. Configuration

Set your date range, API key handling, and feature/model options here.

For local development, use a smaller date range first. For a real training sample, use 2023+.


In [ ]:
# -----------------------
# Core config
# -----------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
PREDICTIONS_DIR = DATA_DIR / "predictions"
for p in [DATA_DIR, MODELS_DIR, PREDICTIONS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TODAY = pd.Timestamp.utcnow().date()
START_DATE = os.getenv("START_DATE", "2023-01-01")
END_DATE = os.getenv("END_DATE", (TODAY + timedelta(days=2)).isoformat())
DAYS_FORWARD_FOR_SCORING = int(os.getenv("DAYS_FORWARD", "2"))

SPORT_KEY = os.getenv("ODDS_SPORT_KEY", "baseball_mlb")
ODDS_REGIONS = os.getenv("ODDS_REGIONS", "us")
ODDS_MARKETS = os.getenv("ODDS_MARKETS", "h2h,spreads,totals")
ODDS_FORMAT = os.getenv("ODDS_FORMAT", "american")
ODDS_API_KEY = os.getenv("ODDS_API_KEY", "")  # Use getpass below if blank.

FETCH_SCHEDULE = True
FETCH_BOXSCORES = True
FETCH_ODDS = True
FETCH_STATCAST = False  # Set True when ready. Statcast pulls can be slow.

STATCAST_CHUNK_DAYS = 7
API_SLEEP_SECONDS = 0.15

MIN_TRAIN_DATE = "2023-01-01"
TEST_FRAC = 0.20
RANDOM_STATE = 42

# Betting thresholds. Tune conservatively at first.
MIN_EDGE_MONEYLINE = 0.02
MIN_EDGE_TOTALS = 0.04
MIN_EDGE_RUNLINE = 0.04
MIN_EV = 0.00

# Export flags default off.
APPROVE_MONEYLINE_EXPORT = False
APPROVE_TOTALS_EXPORT = False
APPROVE_MARGIN_EXPORT = False

print({
    "START_DATE": START_DATE,
    "END_DATE": END_DATE,
    "ODDS_MARKETS": ODDS_MARKETS,
    "FETCH_STATCAST": FETCH_STATCAST,
})


## 2. Utility functions

Reusable helpers for normalization, odds math, chunking, and model evaluation.


In [ ]:
def normalize_team_name(x: Any) -> str:
    if x is None or pd.isna(x):
        return ""
    s = str(x).strip().lower()
    repl = {
        ".": "", "'": "", "&": "and",
        "  ": " ",
    }
    for a, b in repl.items():
        s = s.replace(a, b)
    s = " ".join(s.split())
    aliases = {
        "oakland athletics": "athletics",
        "the athletics": "athletics",
        "la dodgers": "los angeles dodgers",
        "la angels": "los angeles angels",
        "d-backs": "arizona diamondbacks",
        "diamondbacks": "arizona diamondbacks",
        "white sox": "chicago white sox",
        "red sox": "boston red sox",
    }
    return aliases.get(s, s)


def american_to_implied_prob(price: Any) -> float:
    if price is None or pd.isna(price):
        return np.nan
    p = float(price)
    if p > 0:
        return 100.0 / (p + 100.0)
    return abs(p) / (abs(p) + 100.0)


def american_profit_per_unit(price: Any) -> float:
    if price is None or pd.isna(price):
        return np.nan
    p = float(price)
    if p > 0:
        return p / 100.0
    return 100.0 / abs(p)


def expected_value_per_unit(model_prob: float, american_price: float) -> float:
    if pd.isna(model_prob) or pd.isna(american_price):
        return np.nan
    profit = american_profit_per_unit(american_price)
    return model_prob * profit - (1.0 - model_prob)


def no_vig_two_way_prob(price_a: Any, price_b: Any) -> tuple[float, float]:
    pa = american_to_implied_prob(price_a)
    pb = american_to_implied_prob(price_b)
    if pd.isna(pa) or pd.isna(pb) or (pa + pb) <= 0:
        return np.nan, np.nan
    return pa / (pa + pb), pb / (pa + pb)


def daterange_chunks(start_date: str | date, end_date: str | date, chunk_days: int):
    start = pd.to_datetime(start_date).date()
    end = pd.to_datetime(end_date).date()
    cur = start
    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days - 1), end)
        yield cur, chunk_end
        cur = chunk_end + timedelta(days=1)


def rmse(y_true, pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, pred)))


def evaluate_regression(y_true, pred, allow_poisson: bool = False) -> dict[str, float]:
    yt = np.asarray(y_true, dtype=float)
    pr = np.asarray(pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(pr)
    yt = yt[mask]
    pr = pr[mask]
    out = {
        "mae": float(mean_absolute_error(yt, pr)),
        "rmse": float(np.sqrt(mean_squared_error(yt, pr))),
        "r2": float(r2_score(yt, pr)),
        "avg_pred": float(np.mean(pr)),
        "actual_mean": float(np.mean(yt)),
    }
    if allow_poisson:
        pr_pos = np.clip(pr, 1e-6, None)
        yt_pos = np.clip(yt, 0, None)
        try:
            out["poisson_deviance"] = float(mean_poisson_deviance(yt_pos, pr_pos))
        except Exception:
            out["poisson_deviance"] = np.nan
    return out


def evaluate_binary(y_true, prob) -> dict[str, float]:
    yt = np.asarray(y_true, dtype=int)
    pr = np.asarray(prob, dtype=float)
    pr = np.clip(pr, 1e-6, 1 - 1e-6)
    return {
        "n_test": int(len(yt)),
        "avg_pred": float(np.mean(pr)),
        "actual_rate": float(np.mean(yt)),
        "log_loss": float(log_loss(yt, pr)),
        "brier": float(brier_score_loss(yt, pr)),
        "roc_auc": float(roc_auc_score(yt, pr)) if len(np.unique(yt)) == 2 else np.nan,
        "accuracy_50pct": float(accuracy_score(yt, pr >= 0.5)),
    }


def chronological_split(df: pd.DataFrame, test_frac: float = 0.20) -> tuple[pd.DataFrame, pd.DataFrame]:
    sort_cols = [c for c in ["official_date", "game_datetime_utc", "game_pk"] if c in df.columns]
    d = df.sort_values(sort_cols).reset_index(drop=True)
    split_idx = int(len(d) * (1 - test_frac))
    return d.iloc[:split_idx].copy(), d.iloc[split_idx:].copy()

print("Utility functions ready")


## 3. Inline API clients

These cells pull the raw data. Everything is inline: schedule, boxscores, odds, and Statcast.


In [ ]:
def fetch_mlb_schedule(start_date: str, end_date: str, game_type: str = "R", chunk_days: int = 30) -> pd.DataFrame:
    """Fetch MLB games from statsapi.mlb.com schedule endpoint."""
    rows = []
    for s, e in daterange_chunks(start_date, end_date, chunk_days):
        url = "https://statsapi.mlb.com/api/v1/schedule"
        params = {
            "sportId": 1,
            "startDate": s.isoformat(),
            "endDate": e.isoformat(),
            "gameTypes": game_type,
            "hydrate": "probablePitcher,linescore",
        }
        r = requests.get(url, params=params, timeout=45)
        r.raise_for_status()
        data = r.json()
        for dblock in data.get("dates", []) or []:
            for game in dblock.get("games", []) or []:
                teams = game.get("teams", {}) or {}
                home = teams.get("home", {}) or {}
                away = teams.get("away", {}) or {}
                home_team = (home.get("team", {}) or {})
                away_team = (away.get("team", {}) or {})
                status = game.get("status", {}) or {}
                hp = home.get("probablePitcher", {}) or {}
                ap = away.get("probablePitcher", {}) or {}
                rows.append({
                    "game_pk": game.get("gamePk"),
                    "official_date": game.get("officialDate") or dblock.get("date"),
                    "game_datetime_utc": game.get("gameDate"),
                    "game_type": game.get("gameType"),
                    "detailed_state": status.get("detailedState"),
                    "abstract_state": status.get("abstractGameState"),
                    "home_team_id": home_team.get("id"),
                    "home_team_name": home_team.get("name"),
                    "home_team_norm": normalize_team_name(home_team.get("name")),
                    "away_team_id": away_team.get("id"),
                    "away_team_name": away_team.get("name"),
                    "away_team_norm": normalize_team_name(away_team.get("name")),
                    "home_score": home.get("score"),
                    "away_score": away.get("score"),
                    "home_probable_pitcher_id": hp.get("id"),
                    "home_probable_pitcher_name": hp.get("fullName"),
                    "away_probable_pitcher_id": ap.get("id"),
                    "away_probable_pitcher_name": ap.get("fullName"),
                })
        print(f"schedule {s} to {e}: cumulative rows={len(rows)}")
        time.sleep(API_SLEEP_SECONDS)
    df = pd.DataFrame(rows).drop_duplicates("game_pk", keep="last")
    if not df.empty:
        df["official_date"] = pd.to_datetime(df["official_date"], errors="coerce")
        df["game_datetime_utc"] = pd.to_datetime(df["game_datetime_utc"], errors="coerce", utc=True)
        for c in ["home_score", "away_score", "home_team_id", "away_team_id", "home_probable_pitcher_id", "away_probable_pitcher_id"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        df["is_final"] = df["abstract_state"].eq("Final") | df["detailed_state"].astype(str).str.lower().isin(["final", "completed early"])
        df["target_home_win"] = np.where(df["is_final"] & df["home_score"].notna() & df["away_score"].notna(), (df["home_score"] > df["away_score"]).astype(float), np.nan)
        df["target_total_runs"] = np.where(df["is_final"], df["home_score"] + df["away_score"], np.nan)
        df["target_home_margin"] = np.where(df["is_final"], df["home_score"] - df["away_score"], np.nan)
    return df


def fetch_mlb_boxscore(game_pk: int) -> dict:
    url = f"https://statsapi.mlb.com/api/v1/game/{int(game_pk)}/boxscore"
    r = requests.get(url, timeout=45)
    r.raise_for_status()
    return r.json()


def parse_boxscore_team_rows(game_row: pd.Series, box: dict) -> list[dict]:
    """Parse team-level batting/pitching totals from MLB boxscore JSON."""
    out = []
    teams = box.get("teams", {}) or {}
    for side in ["home", "away"]:
        t = teams.get(side, {}) or {}
        team_meta = t.get("team", {}) or {}
        batting = ((t.get("teamStats", {}) or {}).get("batting", {}) or {})
        pitching = ((t.get("teamStats", {}) or {}).get("pitching", {}) or {})
        row = {
            "game_pk": game_row.get("game_pk"),
            "official_date": game_row.get("official_date"),
            "game_datetime_utc": game_row.get("game_datetime_utc"),
            "team_side": side,
            "team_id": team_meta.get("id") or game_row.get(f"{side}_team_id"),
            "team_name": team_meta.get("name") or game_row.get(f"{side}_team_name"),
            "team_norm": normalize_team_name(team_meta.get("name") or game_row.get(f"{side}_team_name")),
            "opponent_team_id": game_row.get("away_team_id" if side == "home" else "home_team_id"),
            "opponent_team_name": game_row.get("away_team_name" if side == "home" else "home_team_name"),
            "runs_for": game_row.get("home_score" if side == "home" else "away_score"),
            "runs_against": game_row.get("away_score" if side == "home" else "home_score"),
        }
        for k, v in batting.items():
            row[f"box_bat_{k}"] = v
        for k, v in pitching.items():
            row[f"box_pitch_{k}"] = v
        out.append(row)
    return out


def fetch_boxscores_for_games(games_df: pd.DataFrame, only_final: bool = True, limit: int | None = None) -> pd.DataFrame:
    candidates = games_df.copy()
    if only_final and "is_final" in candidates.columns:
        candidates = candidates[candidates["is_final"].eq(True)]
    if limit:
        candidates = candidates.head(limit)
    rows = []
    for i, (_, g) in enumerate(candidates.iterrows(), start=1):
        try:
            box = fetch_mlb_boxscore(int(g["game_pk"]))
            rows.extend(parse_boxscore_team_rows(g, box))
        except Exception as exc:
            print(f"boxscore failed game_pk={g.get('game_pk')}: {exc}")
        if i % 100 == 0:
            print(f"boxscores processed {i}/{len(candidates)}")
        time.sleep(API_SLEEP_SECONDS)
    return pd.DataFrame(rows)

print("MLB schedule/boxscore clients ready")


In [ ]:
def fetch_odds_events(api_key: str, sport_key: str = SPORT_KEY, regions: str = ODDS_REGIONS,
                      markets: str = ODDS_MARKETS, odds_format: str = ODDS_FORMAT) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fetch odds from The Odds API and return events + long outcome snapshots."""
    if not api_key:
        raise ValueError("ODDS_API_KEY is blank. Set env var or enter key before running.")
    url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds"
    params = {
        "apiKey": api_key,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
    }
    r = requests.get(url, params=params, timeout=60)
    print("Odds API status:", r.status_code, "remaining:", r.headers.get("x-requests-remaining"), "used:", r.headers.get("x-requests-used"))
    r.raise_for_status()
    data = r.json()
    fetched_at = pd.Timestamp.utcnow().isoformat()
    event_rows = []
    snap_rows = []
    for ev in data:
        event_id = ev.get("id")
        event_rows.append({
            "event_id": event_id,
            "sport_key": ev.get("sport_key"),
            "sport_title": ev.get("sport_title"),
            "commence_time_utc": ev.get("commence_time"),
            "home_team": ev.get("home_team"),
            "away_team": ev.get("away_team"),
            "home_team_norm": normalize_team_name(ev.get("home_team")),
            "away_team_norm": normalize_team_name(ev.get("away_team")),
            "fetched_at_utc": fetched_at,
        })
        for book in ev.get("bookmakers", []) or []:
            for market in book.get("markets", []) or []:
                mkey = market.get("key")
                for outcome in market.get("outcomes", []) or []:
                    snap_rows.append({
                        "fetched_at_utc": fetched_at,
                        "event_id": event_id,
                        "sport_key": ev.get("sport_key"),
                        "commence_time_utc": ev.get("commence_time"),
                        "home_team": ev.get("home_team"),
                        "away_team": ev.get("away_team"),
                        "home_team_norm": normalize_team_name(ev.get("home_team")),
                        "away_team_norm": normalize_team_name(ev.get("away_team")),
                        "bookmaker_key": book.get("key"),
                        "bookmaker_title": book.get("title"),
                        "bookmaker_last_update_utc": book.get("last_update"),
                        "market_key": mkey,
                        "outcome_name": outcome.get("name"),
                        "outcome_name_norm": normalize_team_name(outcome.get("name")),
                        "outcome_price": outcome.get("price"),
                        "outcome_point": outcome.get("point"),
                        "outcome_description": outcome.get("description"),
                    })
    events = pd.DataFrame(event_rows)
    odds = pd.DataFrame(snap_rows)
    for df in [events, odds]:
        if not df.empty and "commence_time_utc" in df.columns:
            df["commence_time_utc"] = pd.to_datetime(df["commence_time_utc"], utc=True, errors="coerce")
    return events, odds

print("Odds API client ready")


In [ ]:
def fetch_statcast_pybaseball(start_date: str, end_date: str, chunk_days: int = 7) -> pd.DataFrame:
    """Fetch pitch-level Statcast using pybaseball.statcast. This is inline and does not use repo scripts."""
    try:
        from pybaseball import statcast
    except Exception as exc:
        raise ImportError("pybaseball is required for Statcast pulls. Run `%pip install pybaseball`.") from exc

    frames = []
    for s, e in daterange_chunks(start_date, end_date, chunk_days):
        print(f"Fetching Statcast {s} to {e}")
        try:
            chunk = statcast(start_dt=s.isoformat(), end_dt=e.isoformat())
            if chunk is not None and len(chunk):
                frames.append(chunk)
                print("  rows", len(chunk))
        except Exception as exc:
            print(f"  Statcast failed {s} to {e}: {exc}")
        time.sleep(API_SLEEP_SECONDS)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df = df.drop_duplicates()
    return df

print("Statcast client ready")


## 4. Pull raw data

Run these cells in order. Start with `FETCH_STATCAST=False` to test schedule/odds quickly. Turn Statcast on when ready.


In [ ]:
if FETCH_SCHEDULE:
    games = fetch_mlb_schedule(START_DATE, END_DATE, game_type="R", chunk_days=30)
else:
    games = pd.DataFrame()

print("games", games.shape)
if not games.empty:
    print(games[["official_date", "game_datetime_utc", "away_team_name", "home_team_name", "detailed_state", "abstract_state", "home_score", "away_score"]].head())
    print("date range", games["official_date"].min(), games["official_date"].max())
    print(games["abstract_state"].value_counts(dropna=False))


In [ ]:
if FETCH_BOXSCORES and not games.empty:
    box_team_game = fetch_boxscores_for_games(games, only_final=True, limit=None)
else:
    box_team_game = pd.DataFrame()

print("box_team_game", box_team_game.shape)
display(box_team_game.head())


In [ ]:
# If ODDS_API_KEY is blank and you are in a notebook, uncomment this:
# import getpass
# ODDS_API_KEY = getpass.getpass("The Odds API key: ")

if FETCH_ODDS and ODDS_API_KEY:
    odds_events, odds_snapshots = fetch_odds_events(ODDS_API_KEY, markets=ODDS_MARKETS)
else:
    print("Skipping odds fetch because FETCH_ODDS is False or ODDS_API_KEY is blank.")
    odds_events, odds_snapshots = pd.DataFrame(), pd.DataFrame()

print("odds_events", odds_events.shape, "odds_snapshots", odds_snapshots.shape)
if not odds_snapshots.empty:
    display(odds_snapshots.groupby("market_key").agg(rows=("event_id", "size"), events=("event_id", "nunique")).reset_index())


In [ ]:
if FETCH_STATCAST:
    statcast_raw = fetch_statcast_pybaseball(START_DATE, END_DATE, chunk_days=STATCAST_CHUNK_DAYS)
else:
    statcast_raw = pd.DataFrame()

print("statcast_raw", statcast_raw.shape)
display(statcast_raw.head())


## 5. Inline feature engineering

This section converts raw schedule/boxscore/Statcast/odds into game-level pregame features. Every rolling feature uses `.shift(1)` so the current game result is not included in its own pregame features.


In [ ]:
def coerce_numeric_cols(df: pd.DataFrame, skip: set[str] | None = None) -> pd.DataFrame:
    out = df.copy()
    skip = skip or set()
    for c in out.columns:
        if c in skip:
            continue
        if out[c].dtype == object:
            # Convert strings like '.321' or '1.234' when possible.
            converted = pd.to_numeric(out[c].astype(str).str.replace("%", "", regex=False), errors="ignore")
            out[c] = converted
    return out


def add_basic_game_outcome_features(games_df: pd.DataFrame) -> pd.DataFrame:
    g = games_df.copy()
    g["home_win"] = np.where(g["is_final"], (g["home_score"] > g["away_score"]).astype(float), np.nan)
    g["away_win"] = np.where(g["is_final"], (g["away_score"] > g["home_score"]).astype(float), np.nan)
    g["home_run_diff"] = np.where(g["is_final"], g["home_score"] - g["away_score"], np.nan)
    g["away_run_diff"] = np.where(g["is_final"], g["away_score"] - g["home_score"], np.nan)
    return g


def team_game_long_from_games(games_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in games_df.iterrows():
        if not bool(r.get("is_final")):
            continue
        for side in ["home", "away"]:
            opp = "away" if side == "home" else "home"
            runs_for = r.get(f"{side}_score")
            runs_against = r.get(f"{opp}_score")
            rows.append({
                "game_pk": r.get("game_pk"),
                "official_date": r.get("official_date"),
                "game_datetime_utc": r.get("game_datetime_utc"),
                "team_side": side,
                "team_id": r.get(f"{side}_team_id"),
                "team_name": r.get(f"{side}_team_name"),
                "team_norm": r.get(f"{side}_team_norm"),
                "opponent_team_id": r.get(f"{opp}_team_id"),
                "opponent_team_name": r.get(f"{opp}_team_name"),
                "runs_for": runs_for,
                "runs_against": runs_against,
                "win": float(runs_for > runs_against) if pd.notna(runs_for) and pd.notna(runs_against) else np.nan,
                "run_diff": runs_for - runs_against if pd.notna(runs_for) and pd.notna(runs_against) else np.nan,
            })
    return pd.DataFrame(rows)


def add_rolling_entity_features(long_df: pd.DataFrame, entity_col: str, value_cols: list[str], prefix: str,
                                windows: list[int] = [3, 5, 10, 20], season: bool = True) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame()
    d = long_df.copy().sort_values([entity_col, "official_date", "game_datetime_utc", "game_pk"])
    out = d[["game_pk", entity_col]].copy()
    for col in value_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce")
        shifted = d.groupby(entity_col)[col].shift(1)
        if season:
            out[f"{prefix}_{col}_season_to_date"] = shifted.groupby(d[entity_col]).expanding(min_periods=1).mean().reset_index(level=0, drop=True)
        for w in windows:
            out[f"{prefix}_{col}_last{w}"] = shifted.groupby(d[entity_col]).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
    return pd.concat([d[["game_pk", entity_col, "official_date", "game_datetime_utc"]].reset_index(drop=True), out.drop(columns=["game_pk", entity_col]).reset_index(drop=True)], axis=1)


def compute_elo_features(games_df: pd.DataFrame, k: float = 20.0, home_adv: float = 35.0, base_elo: float = 1500.0) -> pd.DataFrame:
    g = games_df.copy().sort_values(["official_date", "game_datetime_utc", "game_pk"])
    ratings: dict[int, float] = {}
    rows = []
    for _, r in g.iterrows():
        h = int(r["home_team_id"]) if pd.notna(r.get("home_team_id")) else None
        a = int(r["away_team_id"]) if pd.notna(r.get("away_team_id")) else None
        if h is None or a is None:
            continue
        rh = ratings.get(h, base_elo)
        ra = ratings.get(a, base_elo)
        ph = 1.0 / (1.0 + 10 ** (-((rh + home_adv) - ra) / 400.0))
        rows.append({
            "game_pk": r.get("game_pk"),
            "home_elo_pre": rh,
            "away_elo_pre": ra,
            "diff_elo_pre": rh - ra,
            "elo_home_win_prob": ph,
        })
        if bool(r.get("is_final")) and pd.notna(r.get("home_score")) and pd.notna(r.get("away_score")):
            outcome = 1.0 if r["home_score"] > r["away_score"] else 0.0
            change = k * (outcome - ph)
            ratings[h] = rh + change
            ratings[a] = ra - change
    return pd.DataFrame(rows)

print("Basic rolling/ELO functions ready")


In [ ]:
def prepare_statcast_pitch_level(sc: pd.DataFrame) -> pd.DataFrame:
    if sc.empty:
        return pd.DataFrame()
    d = sc.copy()
    # Normalize important columns.
    rename_map = {"game_date": "official_date"}
    d = d.rename(columns={k: v for k, v in rename_map.items() if k in d.columns})
    if "game_pk" not in d.columns and "game_pk" in d.columns:
        pass
    if "official_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["official_date"], errors="coerce")
    if "game_pk" in d.columns:
        d["game_pk"] = pd.to_numeric(d["game_pk"], errors="coerce")
    for c in ["release_speed", "release_spin_rate", "release_extension", "launch_speed", "launch_angle", "estimated_woba_using_speedangle", "woba_value", "estimated_ba_using_speedangle"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    # batting team and pitching team from inning half.
    if {"inning_topbot", "home_team", "away_team"}.issubset(d.columns):
        d["bat_team"] = np.where(d["inning_topbot"].astype(str).str.lower().eq("top"), d["away_team"], d["home_team"])
        d["pitch_team"] = np.where(d["inning_topbot"].astype(str).str.lower().eq("top"), d["home_team"], d["away_team"])
    d["bat_team_norm"] = d.get("bat_team", pd.Series(index=d.index, dtype=object)).apply(normalize_team_name)
    d["pitch_team_norm"] = d.get("pitch_team", pd.Series(index=d.index, dtype=object)).apply(normalize_team_name)

    # Helpful indicators.
    desc = d.get("description", pd.Series("", index=d.index)).fillna("").astype(str)
    events = d.get("events", pd.Series("", index=d.index)).fillna("").astype(str)
    d["is_pa_event"] = events.ne("")
    d["is_strikeout"] = events.str.contains("strikeout", case=False, na=False)
    d["is_walk"] = events.str.contains("walk", case=False, na=False) & ~events.str.contains("intent", case=False, na=False)
    d["is_home_run"] = events.str.contains("home_run", case=False, na=False)
    d["is_batted_ball"] = d.get("launch_speed", pd.Series(np.nan, index=d.index)).notna()
    d["is_hard_hit"] = d.get("launch_speed", pd.Series(np.nan, index=d.index)).ge(95)
    d["is_sweetspot"] = d.get("launch_angle", pd.Series(np.nan, index=d.index)).between(8, 32)
    d["is_whiff"] = desc.isin(["swinging_strike", "swinging_strike_blocked", "foul_tip"])
    d["is_called_strike"] = desc.eq("called_strike")
    d["is_swing"] = desc.str.contains("swing|foul|hit_into_play", case=False, regex=True, na=False)
    d["pitch_family"] = d.get("pitch_type", pd.Series("UNK", index=d.index)).fillna("UNK").map(pitch_family)
    return d


def pitch_family(pt: Any) -> str:
    if pt is None or pd.isna(pt):
        return "unknown"
    pt = str(pt).upper()
    fast = {"FF", "SI", "FC", "FA", "FS"}
    breaking = {"SL", "CU", "KC", "SV", "ST"}
    offspeed = {"CH", "FS", "FO", "SC"}
    if pt in fast:
        return "fastball"
    if pt in breaking:
        return "breaking"
    if pt in offspeed:
        return "offspeed"
    return "other"


def entropy_from_counts(counts: pd.Series) -> float:
    values = counts[counts > 0].astype(float)
    if values.sum() <= 0:
        return np.nan
    p = values / values.sum()
    return float(-(p * np.log(p)).sum())


def aggregate_statcast_team_game(sc: pd.DataFrame) -> pd.DataFrame:
    if sc.empty:
        return pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    group_cols = ["game_pk", "official_date", "bat_team_norm"]
    agg = d.groupby(group_cols).agg(
        sc_pitches_seen=("game_pk", "size"),
        sc_pa=("is_pa_event", "sum"),
        sc_avg_ev=("launch_speed", "mean"),
        sc_max_ev=("launch_speed", "max"),
        sc_avg_la=("launch_angle", "mean"),
        sc_hard_hit_rate=("is_hard_hit", "mean"),
        sc_sweetspot_rate=("is_sweetspot", "mean"),
        sc_xwoba_contact=("estimated_woba_using_speedangle", "mean"),
        sc_woba=("woba_value", "mean"),
        sc_k_rate=("is_strikeout", "mean"),
        sc_bb_rate=("is_walk", "mean"),
        sc_hr_rate=("is_home_run", "mean"),
        sc_whiff_rate=("is_whiff", "mean"),
        sc_csw_rate=("is_called_strike", "mean"),
    ).reset_index().rename(columns={"bat_team_norm": "team_norm"})
    # CSW should include called strike + whiff over pitches.
    csw = d.assign(csw=d["is_called_strike"] | d["is_whiff"]).groupby(group_cols)["csw"].mean().reset_index(name="sc_csw_rate")
    agg = agg.drop(columns=["sc_csw_rate"], errors="ignore").merge(csw.rename(columns={"bat_team_norm": "team_norm"}), on=["game_pk", "official_date", "team_norm"], how="left")
    return agg


def aggregate_statcast_pitcher_game(sc: pd.DataFrame) -> pd.DataFrame:
    if sc.empty:
        return pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    if "pitcher" not in d.columns:
        return pd.DataFrame()
    group_cols = ["game_pk", "official_date", "pitcher"]
    agg = d.groupby(group_cols).agg(
        sc_pitches=("game_pk", "size"),
        sc_pa=("is_pa_event", "sum"),
        sc_release_speed_mean=("release_speed", "mean"),
        sc_release_spin_mean=("release_spin_rate", "mean"),
        sc_release_extension_mean=("release_extension", "mean"),
        sc_avg_ev_allowed=("launch_speed", "mean"),
        sc_max_ev_allowed=("launch_speed", "max"),
        sc_avg_la_allowed=("launch_angle", "mean"),
        sc_hard_hit_rate_allowed=("is_hard_hit", "mean"),
        sc_sweetspot_rate_allowed=("is_sweetspot", "mean"),
        sc_xwoba_allowed_contact=("estimated_woba_using_speedangle", "mean"),
        sc_woba_allowed=("woba_value", "mean"),
        sc_k_rate=("is_strikeout", "mean"),
        sc_bb_rate=("is_walk", "mean"),
        sc_hr_rate=("is_home_run", "mean"),
        sc_whiff_rate=("is_whiff", "mean"),
    ).reset_index().rename(columns={"pitcher": "pitcher_id"})
    # pitch mix entropy by pitcher-game.
    mix = d.groupby(group_cols + ["pitch_family"]).size().rename("n").reset_index()
    ent = mix.groupby(group_cols)["n"].apply(entropy_from_counts).reset_index(name="sc_pitch_mix_entropy")
    ent = ent.rename(columns={"pitcher": "pitcher_id"})
    agg = agg.merge(ent, on=["game_pk", "official_date", "pitcher_id"], how="left")
    return agg


def aggregate_statcast_pitch_type(sc: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if sc.empty:
        return pd.DataFrame(), pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    team_pt = d.groupby(["game_pk", "official_date", "bat_team_norm", "pitch_family"]).agg(
        pitches=("game_pk", "size"),
        avg_ev=("launch_speed", "mean"),
        woba=("woba_value", "mean"),
        whiff_rate=("is_whiff", "mean"),
        hard_hit_rate=("is_hard_hit", "mean"),
    ).reset_index().rename(columns={"bat_team_norm": "team_norm"})
    pit_pt = d.groupby(["game_pk", "official_date", "pitcher", "pitch_family"]).agg(
        pitches=("game_pk", "size"),
        release_speed=("release_speed", "mean"),
        avg_ev_allowed=("launch_speed", "mean"),
        woba_allowed=("woba_value", "mean"),
        whiff_rate=("is_whiff", "mean"),
        hard_hit_rate_allowed=("is_hard_hit", "mean"),
    ).reset_index().rename(columns={"pitcher": "pitcher_id"})
    return team_pt, pit_pt

print("Statcast feature functions ready")


In [ ]:
def rolling_team_features_from_games(games_df: pd.DataFrame) -> pd.DataFrame:
    long = team_game_long_from_games(games_df)
    if long.empty:
        return pd.DataFrame()
    value_cols = ["runs_for", "runs_against", "win", "run_diff"]
    return add_rolling_entity_features(long, "team_id", value_cols, "team")


def rolling_boxscore_features(box_df: pd.DataFrame) -> pd.DataFrame:
    if box_df.empty:
        return pd.DataFrame()
    d = coerce_numeric_cols(box_df, skip={"team_name", "team_norm", "opponent_team_name", "team_side"})
    value_cols = []
    for c in d.columns:
        if c.startswith("box_bat_") or c.startswith("box_pitch_") or c in ["runs_for", "runs_against"]:
            if pd.api.types.is_numeric_dtype(d[c]):
                value_cols.append(c)
    value_cols = value_cols[:80]  # keep lab manageable; remove cap if desired.
    return add_rolling_entity_features(d, "team_id", value_cols, "box")


def rolling_statcast_team_features(sc_team_game: pd.DataFrame) -> pd.DataFrame:
    if sc_team_game.empty:
        return pd.DataFrame()
    value_cols = [c for c in sc_team_game.columns if c.startswith("sc_")]
    return add_rolling_entity_features(sc_team_game, "team_norm", value_cols, "team_off")


def rolling_statcast_pitcher_features(sc_pitcher_game: pd.DataFrame) -> pd.DataFrame:
    if sc_pitcher_game.empty:
        return pd.DataFrame()
    value_cols = [c for c in sc_pitcher_game.columns if c.startswith("sc_")]
    return add_rolling_entity_features(sc_pitcher_game, "pitcher_id", value_cols, "starter_statcast")


def rolling_pitchmix_team_features(team_pt: pd.DataFrame) -> pd.DataFrame:
    if team_pt.empty:
        return pd.DataFrame()
    # Pivot pitch families wide per game/team.
    piv = team_pt.pivot_table(index=["game_pk", "official_date", "team_norm"], columns="pitch_family", values=["pitches", "avg_ev", "woba", "whiff_rate", "hard_hit_rate"], aggfunc="mean")
    piv.columns = [f"pt_{a}_{b}" for a, b in piv.columns]
    piv = piv.reset_index()
    value_cols = [c for c in piv.columns if c.startswith("pt_")]
    return add_rolling_entity_features(piv, "team_norm", value_cols, "team_pitchmix")

print("Rolling feature functions ready")


In [ ]:
def merge_home_away_team_features(base: pd.DataFrame, feat: pd.DataFrame, entity_col: str, prefix: str, id_home_col: str, id_away_col: str) -> pd.DataFrame:
    if feat.empty:
        return base
    d = base.copy()
    feature_cols = [c for c in feat.columns if c not in {"game_pk", entity_col, "official_date", "game_datetime_utc"}]
    latest = feat[["game_pk", entity_col] + feature_cols].drop_duplicates(["game_pk", entity_col], keep="last")

    home = latest.rename(columns={entity_col: id_home_col, **{c: f"home_{prefix}_{c}" for c in feature_cols}})
    away = latest.rename(columns={entity_col: id_away_col, **{c: f"away_{prefix}_{c}" for c in feature_cols}})

    d = d.merge(home, on=["game_pk", id_home_col], how="left")
    d = d.merge(away, on=["game_pk", id_away_col], how="left")

    for c in feature_cols:
        hc = f"home_{prefix}_{c}"
        ac = f"away_{prefix}_{c}"
        if hc in d.columns and ac in d.columns:
            d[f"diff_{prefix}_{c}"] = d[hc] - d[ac]
    return d


def attach_latest_h2h_odds_features(games_df: pd.DataFrame, odds_df: pd.DataFrame) -> pd.DataFrame:
    d = games_df.copy()
    if odds_df.empty:
        return d
    h2h = odds_df[odds_df["market_key"].astype(str).str.lower().isin(["h2h", "moneyline"])]
    if h2h.empty:
        return d
    event_level = []
    for event_id, ev in h2h.groupby("event_id"):
        ev0 = ev.iloc[0]
        home_norm = ev0["home_team_norm"]
        away_norm = ev0["away_team_norm"]
        home_prices = pd.to_numeric(ev.loc[ev["outcome_name_norm"].eq(home_norm), "outcome_price"], errors="coerce").dropna()
        away_prices = pd.to_numeric(ev.loc[ev["outcome_name_norm"].eq(away_norm), "outcome_price"], errors="coerce").dropna()
        if home_prices.empty or away_prices.empty:
            continue
        hp = float(home_prices.median())
        ap = float(away_prices.median())
        home_nv, away_nv = no_vig_two_way_prob(hp, ap)
        event_level.append({
            "event_id": event_id,
            "odds_commence_time_utc": ev0["commence_time_utc"],
            "home_team_norm": home_norm,
            "away_team_norm": away_norm,
            "home_moneyline_median": hp,
            "away_moneyline_median": ap,
            "market_home_no_vig_prob": home_nv,
            "market_away_no_vig_prob": away_nv,
        })
    evdf = pd.DataFrame(event_level)
    if evdf.empty:
        return d
    # Match by team names and nearest start time within 3 hours.
    rows = []
    for idx, g in d.iterrows():
        cand = evdf[(evdf["home_team_norm"].eq(g.get("home_team_norm"))) & (evdf["away_team_norm"].eq(g.get("away_team_norm")))].copy()
        if cand.empty:
            rows.append({})
            continue
        cand["dt_min"] = (cand["odds_commence_time_utc"] - g["game_datetime_utc"]).abs().dt.total_seconds() / 60.0
        cand = cand.sort_values("dt_min")
        best = cand.iloc[0]
        if best["dt_min"] <= 180:
            rows.append(best.drop(labels=["home_team_norm", "away_team_norm"]).to_dict())
        else:
            rows.append({})
    attach = pd.DataFrame(rows)
    d = pd.concat([d.reset_index(drop=True), attach.reset_index(drop=True)], axis=1)
    d["has_market_odds"] = d.get("home_moneyline_median", pd.Series(np.nan, index=d.index)).notna().astype(int)
    return d


def build_game_feature_frame(games_df: pd.DataFrame, box_df: pd.DataFrame, statcast_raw_df: pd.DataFrame, odds_df: pd.DataFrame) -> pd.DataFrame:
    base = add_basic_game_outcome_features(games_df).copy()
    elo = compute_elo_features(base)
    if not elo.empty:
        base = base.merge(elo, on="game_pk", how="left")

    team_roll = rolling_team_features_from_games(base)
    base = merge_home_away_team_features(base, team_roll, "team_id", "team", "home_team_id", "away_team_id")

    box_roll = rolling_boxscore_features(box_df)
    base = merge_home_away_team_features(base, box_roll, "team_id", "box", "home_team_id", "away_team_id")

    if not statcast_raw_df.empty:
        sc_team = aggregate_statcast_team_game(statcast_raw_df)
        sc_pitcher = aggregate_statcast_pitcher_game(statcast_raw_df)
        team_pt, pit_pt = aggregate_statcast_pitch_type(statcast_raw_df)
        sc_team_roll = rolling_statcast_team_features(sc_team)
        base = merge_home_away_team_features(base, sc_team_roll, "team_norm", "team_sc", "home_team_norm", "away_team_norm")
        pt_roll = rolling_pitchmix_team_features(team_pt)
        base = merge_home_away_team_features(base, pt_roll, "team_norm", "team_pitchmix", "home_team_norm", "away_team_norm")
        sp_roll = rolling_statcast_pitcher_features(sc_pitcher)
        base = merge_home_away_starter_features(base, sp_roll)
    else:
        sc_team = sc_pitcher = team_pt = pit_pt = pd.DataFrame()

    base = attach_latest_h2h_odds_features(base, odds_df)
    return base


def merge_home_away_starter_features(base: pd.DataFrame, sp_feat: pd.DataFrame) -> pd.DataFrame:
    if sp_feat.empty:
        return base
    d = base.copy()
    feature_cols = [c for c in sp_feat.columns if c not in {"game_pk", "pitcher_id", "official_date", "game_datetime_utc"}]
    latest = sp_feat[["game_pk", "pitcher_id"] + feature_cols].drop_duplicates(["game_pk", "pitcher_id"], keep="last")
    home = latest.rename(columns={"pitcher_id": "home_probable_pitcher_id", **{c: f"home_starter_{c}" for c in feature_cols}})
    away = latest.rename(columns={"pitcher_id": "away_probable_pitcher_id", **{c: f"away_starter_{c}" for c in feature_cols}})
    d = d.merge(home, on=["game_pk", "home_probable_pitcher_id"], how="left")
    d = d.merge(away, on=["game_pk", "away_probable_pitcher_id"], how="left")
    for c in feature_cols:
        hc = f"home_starter_{c}"
        ac = f"away_starter_{c}"
        if hc in d.columns and ac in d.columns:
            d[f"diff_starter_{c}"] = d[hc] - d[ac]
    return d

print("Game feature builder ready")


## 6. Build the feature frame

This is the central feature table used for all three tasks.


In [ ]:
features = build_game_feature_frame(games, box_team_game, statcast_raw, odds_snapshots)
print("features", features.shape)
print("date range", features["official_date"].min(), features["official_date"].max())
print("completed rows", features["target_home_win"].notna().sum())
display(features.head())


## 7. Custom feature sandbox

Add new features here. This is the notebook-first version of feature engineering. Once a feature proves useful, port it to production.


In [ ]:
# Example custom interaction features. Add your experiments here.
features = features.copy()

if {"elo_home_win_prob", "market_home_no_vig_prob"}.issubset(features.columns):
    features["diff_elo_minus_market_home_prob"] = features["elo_home_win_prob"] - features["market_home_no_vig_prob"]

# Example: absolute model-independent favorite strength proxies.
if "diff_elo_pre" in features.columns:
    features["abs_diff_elo_pre"] = features["diff_elo_pre"].abs()

print("features after custom sandbox", features.shape)


## 8. Feature inventory, missingness, and leakage audits


In [ ]:
TARGET_HOME_WIN = "target_home_win"
TARGET_TOTAL_RUNS = "target_total_runs"
TARGET_HOME_MARGIN = "target_home_margin"

completed = features[features[TARGET_HOME_WIN].notna()].copy()
completed = completed[pd.to_datetime(completed["official_date"]) >= pd.to_datetime(MIN_TRAIN_DATE)].copy()
print("completed", completed.shape, completed["official_date"].min(), completed["official_date"].max())

NON_FEATURE_COLS_EXACT = {
    "game_pk", "official_date", "game_datetime_utc", "game_type", "detailed_state", "abstract_state", "is_final",
    "home_team_id", "home_team_name", "home_team_norm", "away_team_id", "away_team_name", "away_team_norm",
    "home_probable_pitcher_id", "home_probable_pitcher_name", "away_probable_pitcher_id", "away_probable_pitcher_name",
    "event_id", "odds_commence_time_utc",
    # Hard leakage / targets / final score outcomes.
    "home_score", "away_score", "home_win", "away_win", "home_run_diff", "away_run_diff",
    TARGET_HOME_WIN, TARGET_TOTAL_RUNS, TARGET_HOME_MARGIN,
}
LEAKY_CONTAINS = ["target", "actual_", "final_", "postgame", "post_game", "winner"]


def is_leaky_feature(c: str) -> bool:
    cl = str(c).lower()
    if c in NON_FEATURE_COLS_EXACT:
        return True
    if any(p in cl for p in LEAKY_CONTAINS):
        return True
    if cl.endswith("_score") or cl in {"score", "diff_score", "home_margin", "away_margin"}:
        return True
    return False

numeric_cols = [c for c in completed.columns if pd.api.types.is_numeric_dtype(completed[c])]
clean_feature_cols = [c for c in numeric_cols if not is_leaky_feature(c)]
print("numeric cols", len(numeric_cols), "clean feature cols", len(clean_feature_cols))

feature_inventory = pd.DataFrame({
    "feature": clean_feature_cols,
    "missing_pct": [completed[c].isna().mean() for c in clean_feature_cols],
    "n_unique": [completed[c].nunique(dropna=True) for c in clean_feature_cols],
})
display(feature_inventory.sort_values("missing_pct", ascending=False).head(50))


In [ ]:
def family_filter(cols: list[str], family: str) -> list[str]:
    cl = []
    for c in cols:
        lc = c.lower()
        if family == "all_clean":
            cl.append(c)
        elif family == "market_only":
            if "market" in lc or "moneyline" in lc or "elo" in lc:
                cl.append(c)
        elif family == "team_form":
            if "team_" in lc or "diff_team" in lc or "elo" in lc:
                cl.append(c)
        elif family == "boxscore":
            if "box" in lc:
                cl.append(c)
        elif family == "statcast":
            if "sc_" in lc or "statcast" in lc:
                cl.append(c)
        elif family == "pitchmix":
            if "pitchmix" in lc or "pitch_mix" in lc or "pt_" in lc:
                cl.append(c)
        elif family == "starter":
            if "starter" in lc:
                cl.append(c)
    return cl

feature_sets = {
    "all_clean": family_filter(clean_feature_cols, "all_clean"),
    "team_form": family_filter(clean_feature_cols, "team_form"),
    "boxscore": family_filter(clean_feature_cols, "boxscore"),
    "statcast": family_filter(clean_feature_cols, "statcast"),
    "pitchmix": family_filter(clean_feature_cols, "pitchmix"),
    "starter": family_filter(clean_feature_cols, "starter"),
    "market_plus_elo": family_filter(clean_feature_cols, "market_only"),
}

family_counts = pd.DataFrame([{"feature_family": k, "feature_count": len(v)} for k, v in feature_sets.items()])
display(family_counts.sort_values("feature_count", ascending=False))


In [ ]:
def audit_single_feature_leakage(df: pd.DataFrame, feature_cols: list[str], target_col: str, top_n: int = 40) -> pd.DataFrame:
    rows = []
    y = pd.to_numeric(df[target_col], errors="coerce")
    for c in feature_cols:
        x = pd.to_numeric(df[c], errors="coerce")
        mask = x.notna() & y.notna()
        if mask.sum() < 100 or x[mask].nunique() < 2:
            continue
        xv = x[mask].astype(float)
        yv = y[mask].astype(float)
        corr = np.corrcoef(xv, yv)[0, 1]
        rows.append({
            "feature": c,
            "n": int(mask.sum()),
            "n_unique": int(xv.nunique()),
            "corr_with_target": float(corr),
            "abs_corr": abs(float(corr)),
            "mae_if_direct": float(np.mean(np.abs(yv - xv))),
            "exact_match_rate": float(np.mean(np.isclose(yv, xv, atol=1e-9))),
        })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["exact_match_rate", "abs_corr"], ascending=[False, False]).head(top_n)

print("Moneyline single-feature audit:")
display(audit_single_feature_leakage(completed, clean_feature_cols, TARGET_HOME_WIN, top_n=30))

print("Totals single-feature audit:")
display(audit_single_feature_leakage(completed, clean_feature_cols, TARGET_TOTAL_RUNS, top_n=30))

print("Margin single-feature audit:")
display(audit_single_feature_leakage(completed, clean_feature_cols, TARGET_HOME_MARGIN, top_n=30))


## 9. EDA and optional PCA


In [ ]:
print("Targets")
display(completed[[TARGET_HOME_WIN, TARGET_TOTAL_RUNS, TARGET_HOME_MARGIN]].describe())

# Simple top correlations with targets.
for target in [TARGET_HOME_WIN, TARGET_TOTAL_RUNS, TARGET_HOME_MARGIN]:
    corr_rows = []
    y = pd.to_numeric(completed[target], errors="coerce")
    for c in clean_feature_cols:
        x = pd.to_numeric(completed[c], errors="coerce")
        mask = x.notna() & y.notna()
        if mask.sum() > 100 and x[mask].nunique() > 2:
            corr_rows.append({"feature": c, "corr": np.corrcoef(x[mask], y[mask])[0, 1]})
    corr_df = pd.DataFrame(corr_rows)
    if not corr_df.empty:
        corr_df["abs_corr"] = corr_df["corr"].abs()
        print("
Top correlations with", target)
        display(corr_df.sort_values("abs_corr", ascending=False).head(25))


In [ ]:
RUN_PCA = len(clean_feature_cols) >= 200
PCA_SAMPLE_ROWS = min(len(completed), 5000)
PCA_MAX_FEATURES = min(len(clean_feature_cols), 1000)

if RUN_PCA:
    pca_cols = feature_inventory.sort_values("missing_pct").head(PCA_MAX_FEATURES)["feature"].tolist()
    sample = completed[pca_cols].sample(PCA_SAMPLE_ROWS, random_state=RANDOM_STATE) if len(completed) > PCA_SAMPLE_ROWS else completed[pca_cols]
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=min(50, len(pca_cols)), random_state=RANDOM_STATE)),
    ])
    pipe.fit(sample)
    evr = pipe.named_steps["pca"].explained_variance_ratio_
    pca_summary = pd.DataFrame({"component": np.arange(1, len(evr)+1), "explained_variance_ratio": evr, "cum_explained_variance": np.cumsum(evr)})
    display(pca_summary.head(50))
else:
    print("Skipping PCA; feature count below threshold.")


## 10. Model factories


In [ ]:
def wrap_numeric_model(model, scale: bool = False) -> Pipeline:
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)


def make_classification_models() -> dict[str, Pipeline]:
    models = {
        "logit_l2": wrap_numeric_model(LogisticRegression(max_iter=2000, C=0.5), scale=True),
        "random_forest": wrap_numeric_model(RandomForestClassifier(n_estimators=350, min_samples_leaf=25, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1)),
        "extra_trees": wrap_numeric_model(ExtraTreesClassifier(n_estimators=350, min_samples_leaf=25, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1)),
        "hist_gbdt": wrap_numeric_model(HistGradientBoostingClassifier(max_iter=250, learning_rate=0.035, l2_regularization=0.1, random_state=RANDOM_STATE)),
    }
    if HAS_XGBOOST:
        models["xgboost"] = wrap_numeric_model(XGBClassifier(
            n_estimators=350, max_depth=3, learning_rate=0.025, subsample=0.85, colsample_bytree=0.65,
            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
        ))
    if HAS_LIGHTGBM:
        models["lightgbm"] = wrap_numeric_model(LGBMClassifier(
            n_estimators=400, learning_rate=0.025, num_leaves=24, subsample=0.85, colsample_bytree=0.70,
            random_state=RANDOM_STATE, verbose=-1
        ))
    return models


def make_regression_models(include_poisson: bool = False) -> dict[str, tuple[str, Pipeline]]:
    models: dict[str, tuple[str, Pipeline]] = {
        "random_forest": ("standard_regression", wrap_numeric_model(RandomForestRegressor(n_estimators=350, min_samples_leaf=18, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1))),
        "extra_trees": ("standard_regression", wrap_numeric_model(ExtraTreesRegressor(n_estimators=350, min_samples_leaf=18, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1))),
        "hist_gbdt_squared_error": ("standard_regression", wrap_numeric_model(HistGradientBoostingRegressor(loss="squared_error", max_iter=250, learning_rate=0.035, l2_regularization=0.1, random_state=RANDOM_STATE))),
        "ridge": ("standard_regression", wrap_numeric_model(Ridge(alpha=10.0), scale=True)),
    }
    if HAS_XGBOOST:
        models["xgboost_squarederror"] = ("standard_regression", wrap_numeric_model(XGBRegressor(
            objective="reg:squarederror", n_estimators=350, max_depth=3, learning_rate=0.025,
            subsample=0.85, colsample_bytree=0.65, random_state=RANDOM_STATE, n_jobs=-1
        )))
    if include_poisson:
        models["poisson_glm_l2"] = ("poisson", wrap_numeric_model(PoissonRegressor(alpha=1.0, max_iter=800), scale=True))
        models["hist_gbdt_poisson"] = ("poisson", wrap_numeric_model(HistGradientBoostingRegressor(loss="poisson", max_iter=250, learning_rate=0.035, l2_regularization=0.1, random_state=RANDOM_STATE)))
        if HAS_XGBOOST:
            models["xgboost_poisson"] = ("poisson", wrap_numeric_model(XGBRegressor(
                objective="count:poisson", n_estimators=350, max_depth=3, learning_rate=0.025,
                subsample=0.85, colsample_bytree=0.65, random_state=RANDOM_STATE, n_jobs=-1
            )))
    return models

print("Model factories ready")


## 11. Train/test split


In [ ]:
train_df, test_df = chronological_split(completed, test_frac=TEST_FRAC)
print("Train", train_df.shape, train_df["official_date"].min(), train_df["official_date"].max())
print("Test", test_df.shape, test_df["official_date"].min(), test_df["official_date"].max())


## 12. Moneyline model training


In [ ]:
MODEL_FEATURE_FAMILIES = [k for k, v in feature_sets.items() if len(v) > 0]
print(MODEL_FEATURE_FAMILIES)

y_train_ml = train_df[TARGET_HOME_WIN].astype(int)
y_test_ml = test_df[TARGET_HOME_WIN].astype(int)

moneyline_results = []
moneyline_fitted = {}
moneyline_preds = {}

baseline_rate = float(y_train_ml.mean())
baseline_prob = np.repeat(baseline_rate, len(y_test_ml))
moneyline_results.append({"model_name": "constant_train_home_rate", "feature_count": 0, **evaluate_binary(y_test_ml, baseline_prob)})

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue
    for model_name, estimator in make_classification_models().items():
        full_name = f"{family}__{model_name}"
        print("Training moneyline", full_name, "features", len(cols))
        estimator.fit(train_df[cols], y_train_ml)
        if hasattr(estimator, "predict_proba"):
            prob = estimator.predict_proba(test_df[cols])[:, 1]
        else:
            prob = estimator.predict(test_df[cols])
        metrics = evaluate_binary(y_test_ml, prob)
        moneyline_results.append({"model_name": full_name, "feature_count": len(cols), **metrics})
        moneyline_fitted[full_name] = (estimator, cols)
        moneyline_preds[full_name] = prob

moneyline_results_df = pd.DataFrame(moneyline_results).sort_values(["log_loss", "brier"], na_position="last").reset_index(drop=True)
display(moneyline_results_df.head(30))


## 13. Totals model training


In [ ]:
y_train_total = train_df[TARGET_TOTAL_RUNS].astype(float)
y_test_total = test_df[TARGET_TOTAL_RUNS].astype(float)

totals_results = []
totals_fitted = {}
totals_preds = {}

# Baseline.
baseline_total = np.repeat(y_train_total.mean(), len(y_test_total))
totals_results.append({"model_name": "constant_train_total_runs", "model_objective": "baseline", "feature_count": 0, "n_test": len(y_test_total), **evaluate_regression(y_test_total, baseline_total, allow_poisson=True)})

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue
    for model_name, (objective, estimator) in make_regression_models(include_poisson=True).items():
        full_name = f"{family}__{model_name}"
        print("Training totals", full_name, "features", len(cols))
        estimator.fit(train_df[cols], y_train_total)
        pred = estimator.predict(test_df[cols])
        pred = np.clip(pred, 0.01, None)  # totals cannot be negative.
        metrics = evaluate_regression(y_test_total, pred, allow_poisson=True)
        totals_results.append({"model_name": full_name, "model_objective": objective, "feature_count": len(cols), "n_test": len(y_test_total), **metrics})
        totals_fitted[full_name] = (estimator, cols)
        totals_preds[full_name] = pred

totals_results_df = pd.DataFrame(totals_results).sort_values(["mae", "rmse"], na_position="last").reset_index(drop=True)
display(totals_results_df.head(30))

# Guardrail: suspiciously good totals are probably leakage.
suspicious_totals = totals_results_df[(totals_results_df["mae"] < 2.0) | (totals_results_df["rmse"] < 2.5) | (totals_results_df["r2"] > 0.50)]
if not suspicious_totals.empty:
    print("WARNING: suspicious totals models detected; likely leakage. Do not export these.")
    display(suspicious_totals)


## 14. Home-margin / run-line model training


In [ ]:
y_train_margin = train_df[TARGET_HOME_MARGIN].astype(float)
y_test_margin = test_df[TARGET_HOME_MARGIN].astype(float)

margin_results = []
margin_fitted = {}
margin_preds = {}

baseline_margin = np.repeat(y_train_margin.mean(), len(y_test_margin))
margin_results.append({"model_name": "constant_train_home_margin", "feature_count": 0, "n_test": len(y_test_margin), **evaluate_regression(y_test_margin, baseline_margin)})

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue
    for model_name, (objective, estimator) in make_regression_models(include_poisson=False).items():
        if objective == "poisson":
            continue
        full_name = f"{family}__{model_name}"
        print("Training margin", full_name, "features", len(cols))
        estimator.fit(train_df[cols], y_train_margin)
        pred = estimator.predict(test_df[cols])
        metrics = evaluate_regression(y_test_margin, pred)
        margin_results.append({"model_name": full_name, "feature_count": len(cols), "n_test": len(y_test_margin), **metrics})
        margin_fitted[full_name] = (estimator, cols)
        margin_preds[full_name] = pred

margin_results_df = pd.DataFrame(margin_results).sort_values(["mae", "rmse"], na_position="last").reset_index(drop=True)
display(margin_results_df.head(30))


## 15. Champion selection and residual diagnostics

Manual overrides are intentional. Do not auto-select a suspicious totals model.


In [ ]:
# Recommended defaults: choose the best non-leaky/safe candidates.
# If all_clean totals shows impossible performance, DO NOT use it.
safe_totals_results_df = totals_results_df[
    ~((totals_results_df["mae"] < 2.0) | (totals_results_df["rmse"] < 2.5) | (totals_results_df["r2"] > 0.50))
].copy()

MONEYLINE_CHAMPION_NAME = moneyline_results_df[moneyline_results_df["model_name"].ne("constant_train_home_rate")].iloc[0]["model_name"] if len(moneyline_results_df) > 1 else None
TOTALS_CHAMPION_NAME = safe_totals_results_df[safe_totals_results_df["model_name"].ne("constant_train_total_runs")].iloc[0]["model_name"] if len(safe_totals_results_df) > 1 else None
MARGIN_CHAMPION_NAME = margin_results_df[margin_results_df["model_name"].ne("constant_train_home_margin")].iloc[0]["model_name"] if len(margin_results_df) > 1 else None

# You can override these manually here, e.g.:
# TOTALS_CHAMPION_NAME = "old_statcast_plus_pitchmix_bullpen__random_forest"
# MARGIN_CHAMPION_NAME = "old_statcast_plus_pitchmix_bullpen__random_forest"

print("Moneyline champion:", MONEYLINE_CHAMPION_NAME)
print("Totals champion:", TOTALS_CHAMPION_NAME)
print("Margin champion:", MARGIN_CHAMPION_NAME)

if TOTALS_CHAMPION_NAME:
    display(totals_results_df[totals_results_df["model_name"].eq(TOTALS_CHAMPION_NAME)])
if MARGIN_CHAMPION_NAME:
    display(margin_results_df[margin_results_df["model_name"].eq(MARGIN_CHAMPION_NAME)])


In [ ]:
if TOTALS_CHAMPION_NAME and MARGIN_CHAMPION_NAME:
    total_resid = y_test_total.values - totals_preds[TOTALS_CHAMPION_NAME]
    margin_resid = y_test_margin.values - margin_preds[MARGIN_CHAMPION_NAME]
    total_residual_sigma = float(np.std(total_resid, ddof=1))
    margin_residual_sigma = float(np.std(margin_resid, ddof=1))
    resid_summary = pd.DataFrame([
        {"target": "total_runs", "residual_mean": float(np.mean(total_resid)), "residual_std": total_residual_sigma, "p10": float(np.quantile(total_resid, .10)), "p50": float(np.quantile(total_resid, .50)), "p90": float(np.quantile(total_resid, .90))},
        {"target": "home_margin", "residual_mean": float(np.mean(margin_resid)), "residual_std": margin_residual_sigma, "p10": float(np.quantile(margin_resid, .10)), "p50": float(np.quantile(margin_resid, .50)), "p90": float(np.quantile(margin_resid, .90))},
    ])
    display(resid_summary)
    assert total_residual_sigma > 2.0, "Totals residual sigma is suspiciously low. Check leakage before exporting."
else:
    total_residual_sigma = margin_residual_sigma = np.nan


## 16. Export champion bundles

Exports are off by default. Turn on only after reviewing metrics and leakage audits.


In [ ]:
from datetime import datetime, timezone

if APPROVE_MONEYLINE_EXPORT and MONEYLINE_CHAMPION_NAME:
    assert HAS_JOBLIB, "joblib missing"
    model, cols = moneyline_fitted[MONEYLINE_CHAMPION_NAME]
    bundle = {
        "model_name": MONEYLINE_CHAMPION_NAME,
        "model": model,
        "feature_cols": cols,
        "target_col": TARGET_HOME_WIN,
        "model_type": "moneyline_classification",
        "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "min_train_date": MIN_TRAIN_DATE,
        "probability_shrink": 0.80,
        "shrink_center": 0.50,
    }
    joblib.dump(bundle, MODELS_DIR / "mlb_moneyline_champion.joblib")
    print("Exported", MODELS_DIR / "mlb_moneyline_champion.joblib")
else:
    print("Moneyline export skipped")

if APPROVE_TOTALS_EXPORT and TOTALS_CHAMPION_NAME:
    assert HAS_JOBLIB, "joblib missing"
    model, cols = totals_fitted[TOTALS_CHAMPION_NAME]
    bundle = {
        "model_name": TOTALS_CHAMPION_NAME,
        "model": model,
        "feature_cols": cols,
        "target_col": TARGET_TOTAL_RUNS,
        "model_type": "total_runs_regression",
        "residual_sigma": total_residual_sigma,
        "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "min_train_date": MIN_TRAIN_DATE,
    }
    joblib.dump(bundle, MODELS_DIR / "mlb_total_runs_champion.joblib")
    print("Exported", MODELS_DIR / "mlb_total_runs_champion.joblib")
else:
    print("Totals export skipped")

if APPROVE_MARGIN_EXPORT and MARGIN_CHAMPION_NAME:
    assert HAS_JOBLIB, "joblib missing"
    model, cols = margin_fitted[MARGIN_CHAMPION_NAME]
    bundle = {
        "model_name": MARGIN_CHAMPION_NAME,
        "model": model,
        "feature_cols": cols,
        "target_col": TARGET_HOME_MARGIN,
        "model_type": "home_margin_regression",
        "residual_sigma": margin_residual_sigma,
        "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "min_train_date": MIN_TRAIN_DATE,
    }
    joblib.dump(bundle, MODELS_DIR / "mlb_home_margin_champion.joblib")
    print("Exported", MODELS_DIR / "mlb_home_margin_champion.joblib")
else:
    print("Margin export skipped")


## 17. Inline scoring for upcoming games

This scores future/unfinal games using the in-notebook champion models and current odds. It produces Streamlit-ready dataframes.


In [ ]:
def apply_probability_shrink(p: np.ndarray, shrink: float = 0.80, center: float = 0.50) -> np.ndarray:
    return np.clip(center + shrink * (p - center), 1e-6, 1 - 1e-6)


def score_moneyline(features_df: pd.DataFrame, model_name: str, days_forward: int = 2) -> pd.DataFrame:
    if not model_name:
        return pd.DataFrame()
    model, cols = moneyline_fitted[model_name]
    today = pd.Timestamp.utcnow().date()
    max_date = today + timedelta(days=days_forward)
    cand = features_df[features_df[TARGET_HOME_WIN].isna()].copy()
    cand = cand[(cand["official_date"].dt.date >= today) & (cand["official_date"].dt.date <= max_date)].copy()
    if cand.empty:
        return pd.DataFrame()
    raw = model.predict_proba(cand[cols])[:, 1]
    p_home = apply_probability_shrink(raw, shrink=0.80, center=0.50)
    cand["model_home_win_prob_raw"] = raw
    cand["model_home_win_prob"] = p_home
    cand["model_away_win_prob"] = 1 - p_home
    cand["edge_home"] = cand["model_home_win_prob"] - cand.get("market_home_no_vig_prob", np.nan)
    cand["edge_away"] = cand["model_away_win_prob"] - cand.get("market_away_no_vig_prob", np.nan)
    cand["home_ev_per_unit"] = [expected_value_per_unit(p, price) for p, price in zip(cand["model_home_win_prob"], cand.get("home_moneyline_median", pd.Series(np.nan, index=cand.index)))]
    cand["away_ev_per_unit"] = [expected_value_per_unit(p, price) for p, price in zip(cand["model_away_win_prob"], cand.get("away_moneyline_median", pd.Series(np.nan, index=cand.index)))]

    rec_side = []
    rec_price = []
    rec_prob = []
    rec_market = []
    rec_edge = []
    rec_ev = []
    reason = []
    for _, r in cand.iterrows():
        choices = [
            (r.get("home_team_name"), r.get("home_moneyline_median"), r.get("model_home_win_prob"), r.get("market_home_no_vig_prob"), r.get("edge_home"), r.get("home_ev_per_unit")),
            (r.get("away_team_name"), r.get("away_moneyline_median"), r.get("model_away_win_prob"), r.get("market_away_no_vig_prob"), r.get("edge_away"), r.get("away_ev_per_unit")),
        ]
        choices = [x for x in choices if pd.notna(x[1]) and pd.notna(x[4])]
        if not choices:
            rec_side.append(np.nan); rec_price.append(np.nan); rec_prob.append(np.nan); rec_market.append(np.nan); rec_edge.append(np.nan); rec_ev.append(np.nan); reason.append("no_market_odds"); continue
        best = max(choices, key=lambda x: (x[4], x[5] if pd.notna(x[5]) else -999))
        if best[4] >= MIN_EDGE_MONEYLINE and (pd.isna(best[5]) or best[5] >= MIN_EV):
            rec_side.append(best[0]); rec_price.append(best[1]); rec_prob.append(best[2]); rec_market.append(best[3]); rec_edge.append(best[4]); rec_ev.append(best[5]); reason.append(None)
        else:
            rec_side.append(np.nan); rec_price.append(np.nan); rec_prob.append(np.nan); rec_market.append(np.nan); rec_edge.append(best[4]); rec_ev.append(best[5]); reason.append("below_min_edge")
    cand["recommended_side"] = rec_side
    cand["recommended_price"] = rec_price
    cand["recommended_model_prob"] = rec_prob
    cand["recommended_market_prob"] = rec_market
    cand["edge"] = rec_edge
    cand["expected_value_per_unit"] = rec_ev
    cand["no_bet_reason"] = reason
    return cand

moneyline_predictions = score_moneyline(features, MONEYLINE_CHAMPION_NAME, DAYS_FORWARD_FOR_SCORING) if MONEYLINE_CHAMPION_NAME else pd.DataFrame()
print("moneyline_predictions", moneyline_predictions.shape)
display(moneyline_predictions.head())


In [ ]:
def latest_market_lines(odds_df: pd.DataFrame, market_key: str) -> pd.DataFrame:
    if odds_df.empty:
        return pd.DataFrame()
    d = odds_df[odds_df["market_key"].astype(str).str.lower().eq(market_key)].copy()
    if d.empty:
        return pd.DataFrame()
    # Aggregate by event/outcome/point.
    agg = d.groupby(["event_id", "commence_time_utc", "home_team_norm", "away_team_norm", "outcome_name", "outcome_name_norm", "outcome_point"], dropna=False).agg(
        price_median=("outcome_price", "median"),
        book_count=("bookmaker_key", "nunique"),
    ).reset_index()
    return agg


def attach_totals_spreads_to_games(games_like: pd.DataFrame, odds_df: pd.DataFrame) -> pd.DataFrame:
    out = games_like.copy()
    totals = latest_market_lines(odds_df, "totals")
    spreads = latest_market_lines(odds_df, "spreads")
    # For simplicity, select median total line and over/under prices per event, matched to game by teams/time.
    # Returns columns for market_total_line, over_price_median, under_price_median, home_spread, away_spread, prices.
    total_rows, spread_rows = [], []
    for _, g in out.iterrows():
        # Totals
        tr = {}
        cand = totals[(totals["home_team_norm"].eq(g.get("home_team_norm"))) & (totals["away_team_norm"].eq(g.get("away_team_norm")))].copy()
        if not cand.empty:
            cand["dt_min"] = (cand["commence_time_utc"] - g["game_datetime_utc"]).abs().dt.total_seconds()/60
            cand = cand[cand["dt_min"] <= 180]
            if not cand.empty:
                point = cand["outcome_point"].dropna().median()
                over = cand[cand["outcome_name"].astype(str).str.lower().eq("over")]
                under = cand[cand["outcome_name"].astype(str).str.lower().eq("under")]
                tr = {"market_total_line": point,
                      "over_price_median": over["price_median"].median() if not over.empty else np.nan,
                      "under_price_median": under["price_median"].median() if not under.empty else np.nan}
        total_rows.append(tr)
        # Spreads
        sr = {}
        cand = spreads[(spreads["home_team_norm"].eq(g.get("home_team_norm"))) & (spreads["away_team_norm"].eq(g.get("away_team_norm")))].copy()
        if not cand.empty:
            cand["dt_min"] = (cand["commence_time_utc"] - g["game_datetime_utc"]).abs().dt.total_seconds()/60
            cand = cand[cand["dt_min"] <= 180]
            if not cand.empty:
                home = cand[cand["outcome_name_norm"].eq(g.get("home_team_norm"))]
                away = cand[cand["outcome_name_norm"].eq(g.get("away_team_norm"))]
                sr = {"home_runline_point": home["outcome_point"].median() if not home.empty else np.nan,
                      "home_runline_price": home["price_median"].median() if not home.empty else np.nan,
                      "away_runline_point": away["outcome_point"].median() if not away.empty else np.nan,
                      "away_runline_price": away["price_median"].median() if not away.empty else np.nan}
        spread_rows.append(sr)
    return pd.concat([out.reset_index(drop=True), pd.DataFrame(total_rows), pd.DataFrame(spread_rows)], axis=1)


def score_totals_and_runline(features_df: pd.DataFrame, totals_model_name: str, margin_model_name: str, odds_df: pd.DataFrame, days_forward: int = 2) -> pd.DataFrame:
    today = pd.Timestamp.utcnow().date()
    max_date = today + timedelta(days=days_forward)
    cand = features_df[features_df[TARGET_HOME_WIN].isna()].copy()
    cand = cand[(cand["official_date"].dt.date >= today) & (cand["official_date"].dt.date <= max_date)].copy()
    if cand.empty:
        return pd.DataFrame()
    cand = attach_totals_spreads_to_games(cand, odds_df)

    if totals_model_name:
        model, cols = totals_fitted[totals_model_name]
        cand["model_total_runs"] = np.clip(model.predict(cand[cols]), 0.01, None)
        sig = total_residual_sigma if np.isfinite(total_residual_sigma) else 4.4
        # P(total > market line)
        cand["model_over_prob"] = 1 - norm.cdf(cand["market_total_line"], loc=cand["model_total_runs"], scale=sig)
        cand["model_under_prob"] = 1 - cand["model_over_prob"]
        over_be = cand["over_price_median"].apply(american_to_implied_prob)
        under_be = cand["under_price_median"].apply(american_to_implied_prob)
        cand["edge_over"] = cand["model_over_prob"] - over_be
        cand["edge_under"] = cand["model_under_prob"] - under_be
        cand["over_ev_per_unit"] = [expected_value_per_unit(p, price) for p, price in zip(cand["model_over_prob"], cand["over_price_median"])]
        cand["under_ev_per_unit"] = [expected_value_per_unit(p, price) for p, price in zip(cand["model_under_prob"], cand["under_price_median"])]

    if margin_model_name:
        model, cols = margin_fitted[margin_model_name]
        cand["model_home_margin"] = model.predict(cand[cols])
        sig = margin_residual_sigma if np.isfinite(margin_residual_sigma) else 4.4
        # Home covers if actual_home_margin + home_point > 0 -> actual_home_margin > -home_point
        cand["model_home_runline_cover_prob"] = 1 - norm.cdf(-cand["home_runline_point"], loc=cand["model_home_margin"], scale=sig)
        # Away covers if -actual_home_margin + away_point > 0 -> actual_home_margin < away_point
        cand["model_away_runline_cover_prob"] = norm.cdf(cand["away_runline_point"], loc=cand["model_home_margin"], scale=sig)
        home_be = cand["home_runline_price"].apply(american_to_implied_prob)
        away_be = cand["away_runline_price"].apply(american_to_implied_prob)
        cand["edge_home_runline"] = cand["model_home_runline_cover_prob"] - home_be
        cand["edge_away_runline"] = cand["model_away_runline_cover_prob"] - away_be
        cand["home_runline_ev_per_unit"] = [expected_value_per_unit(p, price) for p, price in zip(cand["model_home_runline_cover_prob"], cand["home_runline_price"])]
        cand["away_runline_ev_per_unit"] = [expected_value_per_unit(p, price) for p, price in zip(cand["model_away_runline_cover_prob"], cand["away_runline_price"])]

    return cand

totals_runline_predictions = score_totals_and_runline(features, TOTALS_CHAMPION_NAME, MARGIN_CHAMPION_NAME, odds_snapshots, DAYS_FORWARD_FOR_SCORING)
print("totals_runline_predictions", totals_runline_predictions.shape)
display(totals_runline_predictions.head())


## 18. Save prediction CSVs for Streamlit


In [ ]:
if not moneyline_predictions.empty:
    out = PREDICTIONS_DIR / "mlb_moneyline_predictions.csv"
    moneyline_predictions.to_csv(out, index=False)
    print("Saved", out)

if not totals_runline_predictions.empty:
    out = PREDICTIONS_DIR / "mlb_totals_runline_predictions.csv"
    totals_runline_predictions.to_csv(out, index=False)
    print("Saved", out)


## 19. Optional GCS upload/download commands

Run these as notebook shell cells if using Colab/Cloud Shell. They are optional; the notebook itself does not depend on files/scripts.


In [ ]:
PROJECT_ID = os.getenv("PROJECT_ID", "tetheredai-preds")
BUCKET = os.getenv("BUCKET", f"tetheredai-mlb-state-{PROJECT_ID}")
print("GCS bucket:", BUCKET)
print("
Example upload commands:")
print(f"gcloud storage cp {PREDICTIONS_DIR / 'mlb_moneyline_predictions.csv'} gs://{BUCKET}/mlb/predictions/mlb_moneyline_predictions.csv")
print(f"gcloud storage cp {PREDICTIONS_DIR / 'mlb_totals_runline_predictions.csv'} gs://{BUCKET}/mlb/predictions/mlb_totals_runline_predictions.csv")
print(f"gcloud storage cp {MODELS_DIR / 'mlb_moneyline_champion.joblib'} gs://{BUCKET}/mlb/models/mlb_moneyline_champion.joblib")
print(f"gcloud storage cp {MODELS_DIR / 'mlb_total_runs_champion.joblib'} gs://{BUCKET}/mlb/models/mlb_total_runs_champion.joblib")
print(f"gcloud storage cp {MODELS_DIR / 'mlb_home_margin_champion.joblib'} gs://{BUCKET}/mlb/models/mlb_home_margin_champion.joblib")


## 20. Production notes

When a feature/model stabilizes in this notebook:

1. Copy the feature engineering code back to production modules.
2. Add tests/leakage audits.
3. Run Cloud Run daily job.
4. Confirm GCS prediction CSVs.
5. Confirm Streamlit update.

This notebook is the one-stop-shop lab; production should stay modular once the model design is stable.
